In [1]:
import time
import chromadb

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.documents import Document

from langchain_chroma import Chroma

from datetime import datetime

In [ ]:


import os
from typing import List, Dict, Any

from langchain.schema import Document
from langchain.text_splitter import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

from langgraph.graph import StateGraph, END


class GraphState(dict):
    """
    State object shared across the entire LangGraph workflow.
    """
    pass


# ============================
# FR‑4: QUERY CLASSIFIER NODE
# ============================




def parallel_retrieval_node(state: GraphState):
    """
    Retrieves top K policy chunks from all required domains in parallel.

    Stores output in:
    state["retrieved_chunks"] = [
        {
            "policy_domain": "",
            "source_file": "",
            "chunk_id": "",
            "content": "",
            "relevance_score": float
        }
    ]
    """

    query = state.get("rewritten_query", state["query"])
    doms = state["query_classification"]["required_policy_domains"]

    results = []

    # We do not restrict domain → simple system
    raw = db.similarity_search_with_score(query, k=5)

    for doc, score in raw:
        results.append({
            "policy_domain": "unknown",   
            "source_file": doc.metadata["source_file"],
            "chunk_id": doc.metadata["chunk_id"],
            "content": doc.page_content,
            "relevance_score": float(score)
        })

    state["retrieved_chunks"] = results
    return state


# ============================
# FR‑7: CONTEXT GRADER NODE
# ============================

def context_grader_node(state: GraphState):
    """
    Grades retrieved chunks as:
    HIGHLY_RELEVANT / PARTIALLY_RELEVANT / WEAK / NOT_RELEVANT

    Implements FR‑7.
    Output:

    state["context_grading"] = {
        "overall_relevance": "",
        "relevant_chunks": [],
        "irrelevant_chunks": [],
        "missing_information": [],
        "decision": "ANSWER | REWRITE_QUERY | ASK_CLARIFICATION | NOT_FOUND"
    }
    """

    chunks = state["retrieved_chunks"]

    relevant = []
    irrelevant = []

    for c in chunks:
        if c["relevance_score"] < 0.6:
            grade = "HIGHLY_RELEVANT"
            relevant.append(c)
        elif c["relevance_score"] < 1.2:
            grade = "PARTIALLY_RELEVANT"
            relevant.append(c)
        else:
            grade = "WEAK"
            irrelevant.append(c)

        c["grade"] = grade

    if len(relevant) == 0:
        decision = "REWRITE_QUERY"
        overall = "NOT_RELEVANT"
    else:
        decision = "ANSWER"
        overall = "RELEVANT"

    state["context_grading"] = {
        "overall_relevance": overall,
        "relevant_chunks": relevant,
        "irrelevant_chunks": irrelevant,
        "missing_information": [],
        "decision": decision
    }

    return state


# ============================
# FR‑8: QUERY REWRITER NODE
# ============================

def query_rewriter_node(state: GraphState):
    """
    Rewrites the query ONLY ONCE if context is weak.
    Implements FR‑8.

    state["rewritten_query"] = "..."
    """

    if state.get("rewrite_attempted", False):
        return state  # already rewritten once

    original = state["query"]

    rewritten = original + " detailed policy rules, exceptions, reimbursement, receipts, limitations"

    state["rewritten_query"] = rewritten
    state["rewrite_attempted"] = True

    return state


# ============================
# ANSWER GENERATOR NODE
# ============================

def answer_generator_node(state: GraphState):
    """
    Generates the answer using relevant chunks only.
    """

    chunks = state["context_grading"]["relevant_chunks"]

    text = "\n".join([c["content"] for c in chunks])

    answer = "Based on relevant policy information:\n\n" + text

    state["draft_answer"] = answer
    return state


# ============================
# FINAL RESPONSE NODE
# ============================

def final_response_node(state: GraphState):
    """
    Produces final answer for the user.
    """
    return {"final_answer": state["draft_answer"]}


# ============================
# BUILD THE GRAPH
# ============================

workflow = StateGraph(GraphState)

workflow.add_node("query_classifier_node", query_classifier_node)
workflow.add_node("parallel_retrieval_node", parallel_retrieval_node)
workflow.add_node("context_grader_node", context_grader_node)
workflow.add_node("query_rewriter_node", query_rewriter_node)
workflow.add_node("answer_generator_node", answer_generator_node)
workflow.add_node("final_response_node", final_response_node)

# entry
workflow.set_entry_point("query_classifier_node")

# edges
workflow.add_edge("query_classifier_node", "parallel_retrieval_node")
workflow.add_edge("parallel_retrieval_node", "context_grader_node")

workflow.add_conditional_edges(
    "context_grader_node",
    lambda state: state["context_grading"]["decision"],
    {
        "ANSWER": "answer_generator_node",
        "REWRITE_QUERY": "query_rewriter_node",
        "ASK_CLARIFICATION": "final_response_node",
        "NOT_FOUND": "final_response_node"
    }
)

workflow.add_edge("query_rewriter_node", "parallel_retrieval_node")
workflow.add_edge("answer_generator_node", "final_response_node")
workflow.add_edge("final_response_node", END)

graph = workflow.compile()


# ============================
# SAMPLE RUN
# ============================

out = graph.invoke({"query": "What is the reimbursement for meals during travel?"})
print(out["final_answer"])


Want Next?
I can give you:
✓ Policy‑domain auto‑mapping (HR, IT_security, Travel, Finance, AI)
✓ Real LLM‑based answer generator (OpenAI, Claude, Groq, Gemini)
✓ Fully production‑grade RAG policy system
✓ Streamlit UI or FastAPI API
✓ Graph visualization
Just tell me what you want next.
